# Submission Packaging and Validation

**Purpose.** Create a clean agent archive from the official runtime and the
reviewed repository policy/deck.

**Decision question.** Is this exact artifact structurally safe and
traceable? This notebook deliberately does not submit automatically; a
human decides whether the evidence deserves one of the latest two slots.

## 1. Discover clean sources

`main.py` and `deck.csv` must sit at the ZIP root, with the complete SDK in
`cg/`. Never package a stale evaluation directory.

In [ ]:
from collections import Counter
from pathlib import Path
import ast
import hashlib
import json
import shutil
import zipfile

PACKAGE_DIR = Path("/kaggle/working/submission_agent")
ARCHIVE = Path("/kaggle/working/submission.zip")

sample = sorted(Path("/kaggle/input").rglob("sample_submission/main.py"))[0].parent
candidates = [Path("../agent"), Path("agent")]
candidates += [x.parent for x in sorted(Path("/kaggle/input").rglob("main.py")) if "sample_submission" not in x.parts and "cg" not in x.parts]
repo_agent = next(
    (x for x in candidates if (x / "main.py").exists() and (x / "deck.csv").exists()),
    None,
)
if repo_agent is None:
    raise FileNotFoundError("Attach a private dataset containing this repo's agent/ directory.")
print(f"Official runtime: {sample}")
print(f"Reviewed agent: {repo_agent}")

## 2. Assemble and statically validate

Start empty so experiment residue cannot leak into the archive. The live
Kaggle validation episode remains authoritative, but syntax, structure,
deck length, and corrupt-ZIP errors are cheaper to catch here.

In [ ]:
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True)
shutil.copytree(sample / "cg", PACKAGE_DIR / "cg")
shutil.copy2(repo_agent / "main.py", PACKAGE_DIR / "main.py")
shutil.copy2(repo_agent / "deck.csv", PACKAGE_DIR / "deck.csv")

source = (PACKAGE_DIR / "main.py").read_text(encoding="utf-8-sig")
tree = ast.parse(source)
names = {node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)}
assert "agent" in names, "main.py must define agent(obs_dict)."

deck = [int(x) for x in (PACKAGE_DIR / "deck.csv").read_text().splitlines() if x.strip()]
assert len(deck) == 60, f"Expected 60 cards, found {len(deck)}."
display(Counter(deck).most_common())

required = ["main.py", "deck.csv", "cg/__init__.py", "cg/api.py", "cg/game.py", "cg/sim.py"]
missing = [x for x in required if not (PACKAGE_DIR / x).exists()]
assert not missing, f"Missing files: {missing}"

## 3. Hash and package the exact artifact

Hashes link ladder results to immutable code and deck content. Archive paths
are relative to staging so no accidental parent folder is introduced.

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "main_sha256": sha256(PACKAGE_DIR / "main.py"),
    "deck_sha256": sha256(PACKAGE_DIR / "deck.csv"),
    "files": sorted(str(x.relative_to(PACKAGE_DIR)).replace("\\", "/")
                    for x in PACKAGE_DIR.rglob("*") if x.is_file()),
}
Path("/kaggle/working/submission_manifest.json").write_text(
    json.dumps(manifest, indent=2)
)
display(manifest)

if ARCHIVE.exists():
    ARCHIVE.unlink()
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PACKAGE_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(PACKAGE_DIR))
with zipfile.ZipFile(ARCHIVE) as archive:
    members, bad = archive.namelist(), archive.testzip()
assert bad is None
assert "main.py" in members and "deck.csv" in members
assert any(x.startswith("cg/") for x in members)
assert not any(x.startswith("submission_agent/") for x in members)
print(f"Archive: {ARCHIVE} ({ARCHIVE.stat().st_size / 1e6:.2f} MB)")
print("\n".join(members))

## 4. Submission gate

Upload only after the evaluation notebook passes, paired evidence beats the
frozen control, hashes are copied to the experiment ledger, current rules
are rechecked, and replacing one of the latest two tracked agents is
intentional. After upload, wait for validation. On failure, download the
agent log and diagnose this exact hash rather than making an unrecorded edit.